In [1]:
import sys 

assert sys.version_info >= (3,10)

In [2]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [4]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("legend", fontsize=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

In [5]:
import deepxde as dde
import numpy as np

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting up the backend


In [6]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")


Set the default float type to float64
Backend: pytorch


Exact solution for validation

In [7]:
def exact_solution(x):
    return (x + 1) ** 2

Domain geometry

In [8]:
geom = dde.geometry.Interval(-1, 1)

Define the Left and Right Boundary Conditions

In [9]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx - 2

def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

def boundary_right(x, on_boudary):
    return on_boudary and np.isclose(x[0], 1)

bc_right = dde.icbc.NeumannBC(geom, lambda x: 4, boundary_right)

Creating data using numpy

In [10]:
observe_x  = np.linspace(-1, 1, 35).reshape(-1, 1)
observe_y = exact_solution(observe_x)
noise = 0.1 * np.random.randn(35, 1)
observe_y = observe_y + noise
observe = dde.icbc.PointSetBC(observe_x, observe_y, component=0)


Combine all data

In [12]:
data = dde.data.PDE(
    geom, 
    pde,
    [bc_left, bc_right, observe],
    num_domain=3000,
    num_boundary=200,
    num_test=500
)

Build a Neural Network and create a Model

In [13]:
net = dde.nn.FNN([1, 256, 128, 64, 1],"tanh", "Glorot uniform")

model = dde.Model(data, net)

Setting Loss Weights

In [14]:
loss_weights = [1.0, 20.0, 1.0, 100.0]
print(f"λ_PDE (Physics):        {loss_weights[0]}")
print(f"λ_BC_left (u(-1)=0):    {loss_weights[1]}")
print(f"λ_BC_right (du/dx=4):   {loss_weights[2]}")
print(f"λ_data (Measurements):  {loss_weights[3]}")

λ_PDE (Physics):        1.0
λ_BC_left (u(-1)=0):    20.0
λ_BC_right (du/dx=4):   1.0
λ_data (Measurements):  100.0


Train the Model with 2 stage optimization
1. Adam Optimizer
2. L-BFGS

In [19]:
print("\nStage 1: Adam Optimizer")
model.compile(
    "adam",
    lr =0.01,
    loss_weights=loss_weights,
    decay=("inverse time", 1000, 0.1)
)
loss_history, train_state = model.train(iterations=2000, display_every=200)

dde.optimizers.config.set_LBFGS_options(maxiter=200)
print("\nStage 2: L-BFGS optinizer (Fine-tuning)")
model.compile("L-BFGS", loss_weights = loss_weights)
loss_history, train_state = model.train(display_every=100)


Stage 1: Adam Optimizer
Compiling model...
'compile' took 0.000306 s

Training model...

Step      Train loss                                  Test loss                                   Test metric
2000      [3.32e-03, 7.08e-04, 1.64e-04, 9.51e-01]    [3.54e-03, 7.08e-04, 1.64e-04, 9.51e-01]    []  
2200      [3.20e-03, 7.52e-04, 1.70e-04, 9.52e-01]    [3.26e-03, 7.52e-04, 1.70e-04, 9.52e-01]    []  
2400      [3.80e-03, 1.69e-04, 4.78e-05, 9.56e-01]    [4.02e-03, 1.69e-04, 4.78e-05, 9.56e-01]    []  
2600      [3.07e-03, 7.12e-04, 1.84e-04, 9.51e-01]    [3.28e-03, 7.12e-04, 1.84e-04, 9.51e-01]    []  
2800      [3.06e-03, 7.12e-04, 1.81e-04, 9.51e-01]    [3.27e-03, 7.12e-04, 1.81e-04, 9.51e-01]    []  
3000      [3.07e-03, 7.11e-04, 1.79e-04, 9.51e-01]    [3.27e-03, 7.11e-04, 1.79e-04, 9.51e-01]    []  
3200      [1.02e-01, 2.28e-02, 7.63e-03, 9.51e-01]    [6.49e-02, 2.28e-02, 7.63e-03, 9.51e-01]    []  
3400      [7.45e-02, 9.71e-03, 1.98e-04, 9.51e-01]    [4.40e-02, 9.71e-03, 1.98

Creating our own test data

In [20]:
test_points = 15
x_test = np.linspace(-1, 1, test_points).reshape(-1,1)
y_pred = model.predict(x_test)
y_exact  = exact_solution(x_test)
noise = 0.1 * np.random.randn(15, 1)
y_test = y_exact + noise

absolute_error = np.abs(y_test - y_pred)
relative_error = absolute_error / (np.abs(y_test) + 1e-10)
l2_error = np.linalg.norm(y_test - y_pred) / np.linalg.norm(y_test)

u_at_minus1 = model.predict(np.array([[-1.0]]))[0, 0]
u_at_plus1  = model.predict(np.array([[1.0]]))[0, 0]
x_right = np.array([[1.0]])
du_dx_at_1 = model.predict(
    x_right,
    operator=lambda x, y: dde.grad.jacobian(y, x, i=0, j=0)
)[0, 0]

print("📊 PERFORMANCE METRICS")
print(f"L2 Relative Error:      {l2_error:.6f}")
print(f"Max Absolute Error:     {np.max(absolute_error):.6f}")
print(f"Mean Absolute Error:    {np.mean(absolute_error):.6f}")

print("\n📍 BOUNDARY CONDITION CHECK")
print(f"u(-1) = {u_at_minus1:.6f}  (Target: 0.0)")
print(f"du/dx(1) = {du_dx_at_1:.6f}  (Target: 4.0)")


📊 PERFORMANCE METRICS
L2 Relative Error:      0.036977
Max Absolute Error:     0.126550
Mean Absolute Error:    0.060206

📍 BOUNDARY CONDITION CHECK
u(-1) = -0.005955  (Target: 0.0)
du/dx(1) = 3.986644  (Target: 4.0)


Now Extracting True Individual Losses

In [21]:
final_train_losses =model.losshistory.loss_train[-1]
L_pde, L_bc_l, L_bc_r, L_data = final_train_losses

print("📉 FINAL TRAINING LOSSES (Unweighted - REAL values used by DeepXDE)")
print(f"L_PDE      : {L_pde:.6e}")
print(f"L_BC_left  : {L_bc_l:.6e}")
print(f"L_BC_right : {L_bc_r:.6e}")
print(f"L_data     : {L_data:.6e}")

print("\n📉 WEIGHTED CONTRIBUTIONS")
weighted = [w * l for w, l in zip(loss_weights, final_train_losses)]
print(f"λ_PDE  × L_PDE      : {weighted[0]:.6e}")
print(f"λ_BC_l × L_BC_left  : {weighted[1]:.6e}")
print(f"λ_BC_r × L_BC_right : {weighted[2]:.6e}")
print(f"λ_data × L_data     : {weighted[3]:.6e}")
print(f"\nTotal Weighted Loss : {sum(weighted):.6e}")


📉 FINAL TRAINING LOSSES (Unweighted - REAL values used by DeepXDE)
L_PDE      : 3.094964e-03
L_BC_left  : 7.093526e-04
L_BC_right : 1.783750e-04
L_data     : 9.513870e-01

📉 WEIGHTED CONTRIBUTIONS
λ_PDE  × L_PDE      : 3.094964e-03
λ_BC_l × L_BC_left  : 1.418705e-02
λ_BC_r × L_BC_right : 1.783750e-04
λ_data × L_data     : 9.513870e+01

Total Weighted Loss : 9.515616e+01
